# Tissue model: semi-supervised, cross-validated

Trains the model that maps the wound bed into **granulation, slough, necrosis, epithelium, callus** (and, where the data
allows, maceration and exposed bone/tendon), the same way the outline model was trained: 5-fold cross-validation (3 if
5 won't fit), grouped by patient, scored on a locked test set.

**Data.** Only ~265 tissue-labelled photos exist publicly: DFUTissue (110, Kaggle), LUTSeg (141 leprosy ulcers, Hugging
Face, CC BY 4.0) and WoundTissue (14 published, GitHub, CC BY-NC-SA). So the training is **semi-supervised**: per fold a
*teacher* learns from the human labels, then labels a pool of **~11,400 other wound photos** (deduplicated: the public
wound datasets + DFUC2022 + post-operative wounds) where it is ≥ 90% sure (pseudo-labels), and a *student* learns from
both. Expect roughly 7,000–9,000 photos to get pseudo-labels per fold (no wound found or too unsure: skipped). Every score
uses human labels only. CO2Wounds is deliberately left out: LUTSeg's photos come from it, so it would leak the test
patients.

**Locked test set:** LUTSeg's 9 gold-standard patients (annotated by 5 clinicians) + DFUTissue's Test split. The summary
compares each class with how well those clinicians agree with each other, and only classes that come close are
*trusted* by the app.

**Before you start (first time only)**
1. Session options: Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only**.
2. **Add Input → Your Work → `notebook2f5893b152`** (your outline-model run: the tissue model starts from its
   `boundary.pt`, and pseudo-labelling uses it to find wounds). If it isn't attached, cell 4 tries to download it.
3. **Turn the session on and run cells 1–5 one by one** (Kaggle only attaches new datasets in an interactive session),
   then the **smoke test** (cell 6, ~10 min). Then **Save Version → Save & Run All (Commit)** for the long run.

Research use only: WoundTissue is non-commercial, DFUTissue states no licence.

## 1. Code

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Your wound photos (the unlabelled pool for pseudo-labels)
Same steps and seed as the outline/wound-type notebook.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
mkdir -p data/raw && cd data/raw
if [ ! -d fuseg ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git src
  (cd src && git sparse-checkout set --no-cone "/data/Foot Ulcer Segmentation Challenge/train/" "/data/Foot Ulcer Segmentation Challenge/validation/")
  mv "src/data/Foot Ulcer Segmentation Challenge" fuseg && rm -rf src
fi
if [ ! -d azh ]; then
  mkdir azh && cd azh
  for f in Train Test; do
    curl -sL -o $f.zip https://raw.githubusercontent.com/uwm-bigdata/wound-classification-using-images-and-locations/main/dataset/$f.zip
    unzip -q $f.zip && rm $f.zip
  done
  rm -rf __MACOSX && cd ..
fi
ls fuseg/train/images | wc -l; ls azh/Train

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
# Same seed and sources as on the Mac, so the splits match and test results are comparable.
python scripts/prepare_data.py \
  --pairs "fuseg:data/raw/fuseg/train/images:data/raw/fuseg/train/labels" \
          "fuseg:data/raw/fuseg/validation/images:data/raw/fuseg/validation/labels" \
  --out data/manifest.csv
python scripts/prepare_data.py \
  --classes "azh:data/raw/azh/Train:wound_type:configs/azh_map.json" "azh:data/raw/azh/Test:wound_type:configs/azh_map.json" \
  --out data/manifest_cls.csv

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/link_kaggle_inputs.py
python scripts/build_public_dataset.py
python scripts/prepare_data.py --csv data/manifest_cls.csv data/public_cls.csv --out data/manifest_cls_v2.csv
python scripts/prepare_data.py --csv data/manifest.csv data/public_seg.csv --out data/manifest_seg_v2.csv
python - <<'EOF'
import pandas as pd
c, s = pd.read_csv('data/manifest_cls_v2.csv'), pd.read_csv('data/manifest_seg_v2.csv')
print(pd.crosstab(c.wound_type, c.split, margins=True))
print(f'wound type: {len(c)} photos (expected 4820) | outline: {len(s)} photos (expected 2582)')
if (len(c), len(s)) != (4820, 2582):
    print('NOTE: counts differ from the local build (a source dataset may have been updated); training continues.')
EOF

## 3. Tissue-labelled data

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/link_kaggle_inputs.py --set tissue          # DFUTissue (Kaggle)
python scripts/link_kaggle_inputs.py --set pool            # extra unlabelled photos: DFUC2022, post-op, FUSeg test
cd data/raw/tissue
[ -d woundtissue ] || git clone -q --depth 1 https://github.com/akabircs/WoundTissue.git woundtissue
python - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download(repo_id="ksanchez84/LUTSeg", repo_type="dataset", local_dir="lutseg")
PY
ls woundtissue/image | wc -l; ls dfutissue/DFUTissue; ls lutseg | head

## 4. The outline model (starting point and wound finder)

In [ ]:
import glob, shutil, os
os.chdir("/kaggle/working/wound_1/wound-ai")
os.makedirs("checkpoints", exist_ok=True)
SOURCE_NOTEBOOK = "vedanshsahay123/notebook2f5893b152"   # your outline-model run
found = sorted(glob.glob("/kaggle/input/**/best/boundary.pt", recursive=True)) or sorted(glob.glob("/kaggle/input/**/boundary.pt", recursive=True))
if not found:
    try:
        import kagglehub
        path = kagglehub.notebook_output_download(SOURCE_NOTEBOOK)
        found = sorted(glob.glob(f"{path}/**/best/boundary.pt", recursive=True))
    except Exception as e:
        print("download failed:", str(e)[:200])
if not found:
    raise SystemExit(f"boundary.pt not found. Add Input -> Your Work -> {SOURCE_NOTEBOOK.split('/')[1]}, then run this cell again.")
shutil.copy(found[0], "checkpoints/boundary.pt")
print("using", found[0])

## 5. Build the tissue dataset
Maps each dataset's labels to the app's classes, locks the test set, builds the unlabelled pool (one photo per near-duplicate group, no copies of labelled photos; expected ~11,360) and measures clinician-to-clinician agreement.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/build_tissue_dataset.py --raw data/raw/tissue \
  --pool data/manifest_cls_v2.csv data/manifest_seg_v2.csv --out data

## 6. Smoke test (~10 min): every stage on a tiny budget

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/tissue_cv.py --manifest data/tissue_manifest.csv --unlabeled data/tissue_unlabeled.csv \
  --boundary checkpoints/boundary.pt --rater data/tissue_rater_agreement.json \
  --folds 2 --smoke --out /kaggle/working/tissue_smoke && rm -rf /kaggle/working/tissue_smoke

## 7. Cross-validation (~8–10 h)
Times a short trial first and prints how many folds it will use (5 if they fit 10 h, else 3). The student sees ~45× more
photos per epoch than the teacher, so 15 epochs (patience 4) is plenty. If the session stops, run the same cell again:
finished steps are kept and training resumes per epoch.

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/tissue_cv.py --manifest data/tissue_manifest.csv --unlabeled data/tissue_unlabeled.csv \
  --boundary checkpoints/boundary.pt --rater data/tissue_rater_agreement.json \
  --folds 5 --fallback-folds 3 --budget-hours 10 --size 512 --batch 8 \
  --teacher-epochs 80 --teacher-patience 15 --student-epochs 15 --student-patience 4 \
  --out /kaggle/working/tissue_cv

## 8. Package the results

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/outputs/best
cd /kaggle/working/tissue_cv
cp summary.md summary.json plan.json /kaggle/working/outputs/
cp best/tissue.pt /kaggle/working/outputs/best/
cp /kaggle/working/wound_1/wound-ai/data/tissue_rater_agreement.json /kaggle/working/outputs/
for d in fold*; do
  cp $d/eval_test.json /kaggle/working/outputs/${d}_test.json 2>/dev/null || true
  cp $d/pseudo/pseudo_stats.json /kaggle/working/outputs/${d}_pseudo.json 2>/dev/null || true
  for m in teacher student; do cp $d/$m/history.csv /kaggle/working/outputs/${d}_${m}_history.csv 2>/dev/null || true; done
done
# Pseudo-label masks: ~100k regenerable files; keep them out of the saved notebook output.
rm -rf /kaggle/working/tissue_cv/fold*/pseudo/masks /kaggle/working/tissue_cv/fold*/pseudo/outlines
cat /kaggle/working/outputs/summary.md
ls -la /kaggle/working/outputs /kaggle/working/outputs/best